# EXP-004 판정 근거

목적: EXP-004(맞은 필드 수가 가장 많은 문서를 앞으로, `filter: best`)의 판정 수치를 SPEC "메타데이터 가설의 판정"(D-18) 순서로 확인하고, multi-doc 상승이 세트 구성 방식(D-13)과 어떻게 이어지는지 본다. 질의 원문은 출력하지 않는다(D-10).

In [1]:
import json
from pathlib import Path

import pandas as pd

from rag.conditions import ConditionIndex
from rag.eval_multi import metrics as multi
from rag.index import load_cfg as load_hydra

In [2]:
ROOT = Path.cwd().parent
RUNS = ROOT / "data/runs"
MULTI = {
    "dense": RUNS / "kure-v1-fixed-512-64-multidoc.csv",
    "exp001": RUNS / "kure-v1-fixed-512-64-hybrid-multidoc.csv",
    "exp003": RUNS / "kure-v1-fixed-512-64-hybrid-cond-multidoc.csv",
    "exp004": RUNS / "kure-v1-fixed-512-64-hybrid-cond-filter-multidoc.csv",
}
COMPARE = {
    "single": RUNS / "exp004-single-compare.json",
    "multi": RUNS / "exp004-multi-compare.json",
}

## 1단계 single-doc

`rag.eval.compare` 출력(기준 EXP-003, 회의 단위 bootstrap 10,000회, seed 20260929).

In [3]:
s = json.loads(COMPARE["single"].read_text(encoding="utf-8"))
{k: s[k] for k in ("diff", "ci", "verdict")}

{'diff': -0.0003315649867374005,
 'ci': [-0.0010452961672473868, 0.0],
 'verdict': '보류'}

1단계가 보류라 2단계 multi-doc으로 정한다.

## 2단계 multi-doc

`rag.eval_multi.compare` 출력(기준 EXP-003, `pool_id` 단위 bootstrap 10,000회, 동결 세트 600건).

In [4]:
m = json.loads(COMPARE["multi"].read_text(encoding="utf-8"))
diffs = pd.DataFrame([r for v in m.values() if isinstance(v, list) for r in v])
diffs["diff %p"] = (diffs["diff"] * 100).round(2)
diffs["95% 구간 %p"] = diffs["ci"].map(lambda c: [round(x * 100, 2) for x in c])
keep = diffs["metric"].isin(["recall@5", "complete@5", "hit@5"])
diffs.loc[keep, ["group", "value", "n", "metric", "diff %p", "95% 구간 %p"]]

,group,value,n,metric,diff %p,95% 구간 %p
0,all,all,600,recall@5,35.54,"[32.24, 38.81]"
1,all,all,600,complete@5,44.00,"[39.83, 48.0]"
2,all,all,600,hit@5,25.33,"[21.83, 28.83]"
6,type,conf,233,recall@5,6.20,"[3.97, 8.58]"
7,type,conf,233,complete@5,13.30,"[8.58, 18.03]"
8,type,conf,233,hit@5,0.43,"[0.0, 1.29]"
12,type,law,152,recall@5,38.78,"[32.84, 44.82]"
13,type,law,152,complete@5,44.74,"[36.18, 53.29]"
14,type,law,152,hit@5,27.63,"[20.39, 34.87]"
18,type,questioner,215,recall@5,65.03,"[59.57, 70.45]"


## multi-doc 순위 네 개 비교

In [5]:
hcfg = load_hydra(["+exp=exp004", "index.scope=full", "search.queries=multidoc"])
ev = hcfg.multidoc.eval
gold = multi.load_gold(ROOT / hcfg.paths.multidoc_frozen)
scores = {
    name: multi.per_query(multi.ranked_docs(multi.read_run(p), gold, max(ev.ks)), gold, ev.ks)
    for name, p in MULTI.items()
}
cols = ["recall@5", "complete@5"]
pd.concat({name: s[cols].join(gold["type"]).groupby("type").mean()
           for name, s in scores.items()}, axis=1).round(4)

dense              exp001              exp003             \
           recall@5 complete@5 recall@5 complete@5 recall@5 complete@5   
type                                                                     
conf         0.8288     0.6567   0.8059     0.6223   0.9207     0.8197   
law          0.2990     0.1053   0.2231     0.0855   0.4484     0.2039   
questioner   0.1900     0.0930   0.1790     0.0930   0.3060     0.1721   

             exp004             
           recall@5 complete@5  
type                            
conf         0.9827     0.9528  
law          0.8362     0.6513  
questioner   0.9564     0.9395

## 최다 일치 문서와 후보 집합

multi-doc 세트는 메타데이터 값(회의, 법안, 질의자·위원회)으로 후보 집합을 묶고, 질의는 그 집합에서 고른 정답 문서를 날짜·위원회·질의자 이름으로 지목한다(D-13). 질의마다 필터가 앞으로 옮기는 최다 일치 문서 집합이 정답 문서를 얼마나 담는지, 그 집합이 후보 집합 안에 얼마나 있는지 본다.

In [6]:
with (ROOT / hcfg.paths.corpus).open(encoding="utf-8") as f:
    docs = list(map(json.loads, f))
idx = ConditionIndex(docs, hcfg.retriever.conditions.year_ref_offsets)
names = [d["doc_id"] for d in docs]
with (ROOT / hcfg.paths.conditions).open(encoding="utf-8") as f:
    conds = {r["qid"]: r["conditions"] for r in map(json.loads, f)}
with (ROOT / hcfg.paths.multidoc_pools).open(encoding="utf-8") as f:
    pools = {r["pool_id"]: set(r["doc_ids"]) for r in map(json.loads, f)}
rows = []
for q, g in gold.iterrows():
    counts = idx.match_counts(conds[q])
    top = max(counts.values())
    best = {names[i] for i, c in counts.items() if c == top}
    rows.append({"qid": q, "type": g["type"], "최다 일치 문서 수": len(best),
                 "정답 중 최다 일치 비율": len(g["gold"] & best) / len(g["gold"]),
                 "최다 일치 중 후보 집합 비율": len(best & pools[g["pool_id"]]) / len(best)})
best_df = pd.DataFrame(rows).set_index("qid")
best_df.groupby("type").agg(["mean", "median"]).round(3)

최다 일치 문서 수        정답 중 최다 일치 비율        최다 일치 중 후보 집합 비율       
                 mean median          mean median             mean median
type                                                                     
conf           12.215   12.0         0.987    1.0            0.939    1.0
law            11.132    4.5         0.879    1.0            0.579    0.5
questioner      4.098    2.0         0.971    1.0            0.950    1.0

In [7]:
print("정답이 모두 최다 일치 집합에 든 질의 비율")
(best_df["정답 중 최다 일치 비율"] == 1).groupby(best_df["type"]).mean().round(3)

정답이 모두 최다 일치 집합에 든 질의 비율


type
conf          0.970
law           0.763
questioner    0.944
Name: 정답 중 최다 일치 비율, dtype: float64